# Digital DNA — 02: Feature Analysis
Feature correlation, variance, and discriminative separation between genuine and impostor behavioral profiles.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from features.feature_pipeline import FeaturePipeline
print("Keystroke Features:", len(FeaturePipeline.get_feature_names("keystroke")))
print("Mouse Features:", len(FeaturePipeline.get_feature_names("mouse")))

## 1. Feature Correlation Analysis

In [ ]:
k_df = pd.read_csv(project_root / "data" / "processed" / "benchmark_keystroke.csv")
k_features = [c for c in FeaturePipeline.get_feature_names("keystroke") if c in k_df.columns]

corr_matrix = k_df[k_features].corr()
print("Top correlated feature pairs in Keystroke dynamics:")
unstack = corr_matrix.abs().unstack()
pairs = unstack[unstack < 1.0].sort_values(ascending=False).drop_duplicates().head(10)
print(pairs)

## 2. Feature Discriminative Power (Cohen's d Separation)

In [ ]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    s1, s2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
    return abs(np.mean(group1) - np.mean(group2)) / (pooled_std + 1e-9)

gen = k_df[k_df["label"] == 1]
imp = k_df[k_df["label"] == 0]

effect_sizes = {feat: cohens_d(gen[feat], imp[feat]) for feat in k_features}
sorted_effects = sorted(effect_sizes.items(), key=lambda x: x[1], reverse=True)

print("Keystroke Features ranked by discriminative effect size (Cohen's d):")
for feat, d in sorted_effects:
    print(f"  {feat:25s}: d = {d:.2f}")